# 4단계: 문장 선택 (Colab, GPU)

3단계에서 재정렬한 **top-5 문서**(원본 HoVer와 같음)의 모든 문장을 후보로 두고, (claim, 문장) 쌍마다 "근거 문장일 확률"을 매겨 상위 문장을 고른다.

- 정답 = `supporting_facts`의 (문서 제목, 문장 번호)
- 문장 번호를 맞추려고 0단계와 똑같이 CoreNLP로 문장을 나눈다
- 평가: 고른 top-k 문장 안에 정답 근거 문장이 **모두** 들어왔는가 (+ 문장 단위 recall/precision)
- 천장: 문서 단계 recall@5 (test 0.558). 문서를 놓치면 그 안의 문장도 고를 수 없다

| 셀 | 역할 |
|---|---|
| 1 | 0단계 데이터 + 3단계 재정렬 결과 불러오기 |
| 2 | CoreNLP 서버 시작, 문서 → 문장 함수 |
| 3 | (claim, 문장) 쌍 만들기. 학습 때는 정답 문서를 항상 포함 (3단계와 같은 이유) |
| 4 | 문장 선택 모델 학습 |
| 5 | 문장 점수 매기기, k별 평가 → val에서 k를 고른다 |
| 6 | 결과 저장 (5단계에서 씀) |

In [9]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U transformers datasets accelerate stanza

import json, os, sqlite3, unicodedata
from functools import lru_cache
import numpy as np, pandas as pd

ROOT = '/content/drive/MyDrive/fact-verification'
RAW, PROC = f'{ROOT}/data/raw', f'{ROOT}/data/processed'
WIKI = '/content/wiki_wo_links.db'
if not os.path.exists(WIKI):
    !cp "{RAW}/wiki_wo_links.db" {WIKI}
con = sqlite3.connect(WIKI)

nfc = lambda s: unicodedata.normalize('NFC', s)  # 정답 제목은 NFC, 위키 DB/TF-IDF 제목은 NFD라서 비교 전에 통일
TAG = '_mh'  # 베이스라인: '' / 08 multi-hop 문서 검색 결과를 쓰려면 '_mh'
dfs = {}
for n in ['train', 'val', 'test']:
    d = pd.read_json(f'{PROC}/hover2_{n}.jsonl', lines=True)
    r = pd.read_json(f'{PROC}/hover2_{n}_docret{TAG}.jsonl', lines=True)
    d = d.merge(r, on='uid', validate='1:1')
    d['reranked'] = d.reranked.map(lambda ts: [nfc(t) for t in ts])
    d['gold_docs'] = d.supporting_facts.map(lambda sf: list(dict.fromkeys(nfc(t) for t, _ in sf)))
    d['gold_sents'] = d.supporting_facts.map(lambda sf: {(nfc(t), i) for t, i in sf})
    dfs[n] = d
{n: len(d) for n, d in dfs.items()}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


{'train': 8166, 'val': 886, 'test': 1126}

In [10]:
!which java || apt-get -qq install -y openjdk-17-jre-headless
import stanza
from stanza.server import CoreNLPClient, StartServer

os.environ['CORENLP_HOME'] = '/content/corenlp'
if not os.path.exists('/content/corenlp'):
    stanza.install_corenlp(dir='/content/corenlp')
# 9000번 포트는 Colab에서 뜨지 않았다. 문서 수만 개를 처리하므로 로그는 끈다
# TRY_START: 앞서 띄운 서버가 9011에 남아 있으면 새로 띄우지 않고 그 서버에 연결한다
client = CoreNLPClient(annotators=['tokenize', 'ssplit'], timeout=60000, memory='4G',
                       be_quiet=True, endpoint='http://localhost:9011', start_server=StartServer.TRY_START)
client.start()

@lru_cache(maxsize=None)
def get_sents(title):
    row = con.execute('SELECT text FROM documents WHERE id = ?', (unicodedata.normalize('NFD', title),)).fetchone()
    if row is None:
        return ()
    para = row[0]
    ann = client.annotate(para, output_format='json')
    return tuple(para[s['tokens'][0]['characterOffsetBegin']:s['tokens'][-1]['characterOffsetEnd']]
                 for s in ann['sentences'])

get_sents('Peder Severin Krøyer')

/usr/bin/java


INFO:stanza:Installing CoreNLP 4.6.0 into /content/corenlp


stanford-corenlp-latest.zip: reconstructing file:   0%|          |  0.00B /  511MB            

stanford-corenlp-latest.zip: downloading bytes:           |  0.00B            

INFO:stanza:Downloaded file to /content/corenlp/corenlp.zip
INFO:stanza:Installed CoreNLP 4.6.0 into /content/corenlp
INFO:stanza:Writing properties to tmp file: corenlp_server-b3f778edb01f4f31.props
INFO:stanza:Starting server with command: java -Xmx4G -cp /content/corenlp/* edu.stanford.nlp.pipeline.StanfordCoreNLPServer -port 9011 -timeout 60000 -threads 5 -maxCharLength 100000 -quiet True -serverProperties corenlp_server-b3f778edb01f4f31.props -annotators tokenize,ssplit -preload -outputFormat serialized


("Peder Severin Krøyer ['peːdə 'seveʁin 'kʁojə] (23 July 1851 – 21 November 1909), professionally known as P. S. Krøyer, was a Danish painter.",
 'He is one of the best known and beloved, and the most colorful of the Skagen Painters, a community of Danish and Nordic artists who lived, gathered, or worked in Skagen, Denmark, especially during the final decades of the 19th century.',
 'Krøyer was the unofficial leader of the group.')

In [11]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL = 'roberta-base'
MAX_LEN = 128
K_DOC = 5
tok = AutoTokenizer.from_pretrained(MODEL)

def make_pairs(d, train):
    rows = []
    for r in d.itertuples():
        titles = list(r.reranked[:K_DOC])
        if train:
            titles = (r.gold_docs + [t for t in titles if t not in r.gold_docs])[:K_DOC]
        for t in titles:
            for i, s in enumerate(get_sents(t)):
                rows.append({'uid': r.uid, 'claim': r.claim, 'title': t, 'sid': i, 'sent': s,
                             'labels': int((t, i) in r.gold_sents)})
    return pd.DataFrame(rows)

pairs = {n: make_pairs(d, n == 'train') for n, d in dfs.items()}

def encode(b):
    # 문장만으로는 "He", "It" 같은 대명사가 누구인지 모르니 문서 제목을 앞에 붙인다
    return tok(b['claim'], [f'{t}. {s}' for t, s in zip(b['title'], b['sent'])], truncation=True, max_length=MAX_LEN)  # 긴 쪽부터 자른다 (claim이 길어도 오류 없음)

ds = {n: Dataset.from_pandas(p[['claim', 'title', 'sent', 'labels']]).map(encode, batched=True, remove_columns=['claim', 'title', 'sent'])
      for n, p in pairs.items()}
{n: (len(p), round(float(p.labels.mean()), 3)) for n, p in pairs.items()}  # (쌍 개수, 양성 비율)

Map:   0%|          | 0/142746 [00:00<?, ? examples/s]

Map:   0%|          | 0/14511 [00:00<?, ? examples/s]

Map:   0%|          | 0/18571 [00:00<?, ? examples/s]

{'train': (142746, 0.139), 'val': (14511, 0.138), 'test': (18571, 0.137)}

In [12]:
import torch
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
args = TrainingArguments(
    output_dir='/content/out/sent_select',
    num_train_epochs=1, learning_rate=2e-5, per_device_train_batch_size=32, per_device_eval_batch_size=256,
    warmup_steps=0.1, weight_decay=0.01, fp16=True, seed=42,
    save_strategy='no', logging_steps=200, report_to='none',
)
trainer = Trainer(model=model, args=args, train_dataset=ds['train'], data_collator=DataCollatorWithPadding(tok))
trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
200,0.495454
400,0.301693
600,0.276332
800,0.281453
1000,0.239967
1200,0.239306
1400,0.225962
1600,0.234885
1800,0.210321
2000,0.215807


TrainOutput(global_step=4461, training_loss=0.23064652164803842, metrics={'train_runtime': 750.6782, 'train_samples_per_second': 190.156, 'train_steps_per_second': 5.943, 'total_flos': 8530652414645760.0, 'train_loss': 0.23064652164803842, 'epoch': 1.0})

In [13]:
# 웹 데모(app.py)에서 쓰도록 학습한 모델을 Drive에 저장한다
SAVE = '/content/drive/MyDrive/fact-verification/models/sent_select'
trainer.save_model(SAVE); tok.save_pretrained(SAVE)
!ls -lh {SAVE}

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

total 479M
-rw------- 1 root root  740 Oct 27 04:46 config.json
-rw------- 1 root root 476M Oct 27 04:46 model.safetensors
-rw------- 1 root root  388 Oct 27 04:46 tokenizer_config.json
-rw------- 1 root root 3.4M Oct 27 04:46 tokenizer.json
-rw------- 1 root root 5.1K Oct 27 04:46 training_args.bin


In [14]:
TOP_SAVE = 10  # 5단계에서 k를 바꿔 볼 수 있게 넉넉히 저장
N_DIV = 2  # 서로 다른 문서 N_DIV개의 1위 문장을 맨 앞에 둔다 (0이면 점수 순 그대로). 정답 문서마다 근거가 1개는 들어가게 하려는 것

def score(n):
    p = pairs[n].copy()
    logits = trainer.predict(ds[n]).predictions
    p['score'] = torch.softmax(torch.tensor(logits), -1)[:, 1].numpy()
    return p.sort_values('score', ascending=False)

def rank(n, n_div=N_DIV):
    p = scored[n]
    first = p.drop_duplicates(['uid', 'title']).groupby('uid', sort=False).head(n_div).index  # 문서별 1위 문장 중 상위 n_div개
    p = p.assign(first=p.index.isin(first)).sort_values(['first', 'score'], ascending=False, kind='stable')
    p = p.groupby('uid', sort=False).head(TOP_SAVE)
    ranked = {uid: g[['title', 'sid', 'sent', 'score']].values.tolist() for uid, g in p.groupby('uid', sort=False)}
    return dfs[n].uid.map(lambda u: ranked.get(u, []))

def sent_metrics(d, k):
    allin, cover, rec, prec = [], [], [], []
    for gold, pred in zip(d.gold_sents, d.pred_sents):
        top = {(t, int(i)) for t, i, *_ in pred[:k]}
        hit = len(gold & top)
        allin.append(gold <= top); rec.append(hit / len(gold)); prec.append(hit / max(len(top), 1))
        cover.append(all(any(g in top for g in gold if g[0] == doc) for doc in {t for t, _ in gold}))  # 정답 문서마다 정답 문장 1개 이상 (공식 HoVer 조건)
    return {'all_gold': round(float(np.mean(allin)), 3), 'doc_cover': round(float(np.mean(cover)), 3),
            'recall': round(float(np.mean(rec)), 3), 'precision': round(float(np.mean(prec)), 3)}

scored = {n: score(n) for n in ['train', 'val', 'test']}

# N_DIV는 val로 고른다
for nd in [0, 2, 3]:
    d = dfs['val'].assign(pred_sents=rank('val', nd))
    print(f'val N_DIV={nd}', {k: (m['all_gold'], m['doc_cover']) for k in [2, 3, 5] for m in [sent_metrics(d, k)]}, '(all_gold, doc_cover)')

for n in ['train', 'val', 'test']:
    dfs[n]['pred_sents'] = rank(n)

for n in ['val', 'test']:
    d = dfs[n]
    ceiling = np.mean([set(g) <= set(r[:K_DOC]) for r, g in zip(d.reranked, d.gold_docs)])
    print(n, f'N_DIV={N_DIV}, 천장(문서 recall@{K_DOC}) {ceiling:.3f}')
    for k in [2, 3, 4, 5]:
        print(f'  top-{k}', sent_metrics(d, k))

val N_DIV=0 {2: (0.439, 0.589), 3: (0.581, 0.738), 5: (0.757, 0.853)} (all_gold, doc_cover)
val N_DIV=2 {2: (0.468, 0.64), 3: (0.585, 0.743), 5: (0.758, 0.854)} (all_gold, doc_cover)
val N_DIV=3 {2: (0.468, 0.64), 3: (0.507, 0.714), 5: (0.745, 0.852)} (all_gold, doc_cover)
val N_DIV=2, 천장(문서 recall@5) 0.897
  top-2 {'all_gold': 0.468, 'doc_cover': 0.64, 'recall': 0.709, 'precision': 0.797}
  top-3 {'all_gold': 0.585, 'doc_cover': 0.743, 'recall': 0.795, 'precision': 0.609}
  top-4 {'all_gold': 0.678, 'doc_cover': 0.807, 'recall': 0.843, 'precision': 0.491}
  top-5 {'all_gold': 0.758, 'doc_cover': 0.854, 'recall': 0.882, 'precision': 0.415}
test N_DIV=2, 천장(문서 recall@5) 0.875
  top-2 {'all_gold': 0.399, 'doc_cover': 0.577, 'recall': 0.665, 'precision': 0.759}
  top-3 {'all_gold': 0.539, 'doc_cover': 0.682, 'recall': 0.763, 'precision': 0.594}
  top-4 {'all_gold': 0.64, 'doc_cover': 0.758, 'recall': 0.823, 'precision': 0.487}
  top-5 {'all_gold': 0.709, 'doc_cover': 0.803, 'recall': 0.85

In [15]:
import collections

# 문서는 top-5에 다 들어왔는데 정답 문장을 놓친 claim을 원인별로 나눈다
# 문장 k=5 = 07 검증기가 보는 근거 기준. 공식 HoVer 채점(상위 10문장, 앞 3문서, 문서마다 2문장)과는 다르다
# '근거가 하나도 없는 정답 문서가 있음'이 많으면 → 문서마다 최소 1문장 고르기
# '문서마다 1개 이상은 찾음'이 많으면 → 정답 문장이 3개 이상인 claim. k를 늘리거나 문장에 맥락 붙이기
def miss_type(gold, pred, k=5):
    top = {(t, int(i)) for t, i, *_ in pred[:k]}
    if gold <= top:
        return '문장 다 찾음'
    if all(any(g in top for g in gold if g[0] == doc) for doc in {t for t, _ in gold}):
        return '문서마다 1개 이상은 찾음'
    return '근거가 하나도 없는 정답 문서가 있음'

for n in ['val', 'test']:  # 방법은 val로 고른다
    d = dfs[n]
    ok = [set(g) <= set(r[:K_DOC]) for r, g in zip(d.reranked, d.gold_docs)]
    print(n, f'문서 top-{K_DOC} 성공 {sum(ok)}건', dict(collections.Counter(miss_type(g, p) for o, g, p in zip(ok, d.gold_sents, d.pred_sents) if o)))


val 문서 top-5 성공 795건 {'문장 다 찾음': 672, '문서마다 1개 이상은 찾음': 85, '근거가 하나도 없는 정답 문서가 있음': 38}
test 문서 top-5 성공 985건 {'문장 다 찾음': 798, '근거가 하나도 없는 정답 문서가 있음': 81, '문서마다 1개 이상은 찾음': 106}


In [16]:
for n, d in dfs.items():
    d[['uid', 'pred_sents']].to_json(f'{PROC}/hover2_{n}_sentret{TAG}.jsonl', orient='records', lines=True, force_ascii=False)
!ls -lh "{PROC}" 

total 67M
-rw------- 1 root root 4.0M Oct 27 03:13 hover2_test_docret.jsonl
-rw------- 1 root root 525K Oct 27 04:09 hover2_test_docret_mh.jsonl
-rw------- 1 root root 2.4M Oct 21 13:28 hover2_test_e2e_retrieved.jsonl
-rw------- 1 root root 2.3M Oct 27 02:29 hover2_test_e2e_retrieved_mh.jsonl
-rw------- 1 root root 768K Oct 21 06:22 hover2_test.jsonl
-rw------- 1 root root 2.1M Oct 21 12:55 hover2_test_sentret.jsonl
-rw------- 1 root root 2.0M Oct 27 04:50 hover2_test_sentret_mh.jsonl
-rw------- 1 root root 6.3M Oct 27 03:13 hover2_train_docret.jsonl
-rw------- 1 root root 4.2M Oct 27 04:09 hover2_train_docret_mh.jsonl
-rw------- 1 root root 5.5M Oct 21 06:22 hover2_train.jsonl
-rw------- 1 root root  15M Oct 21 12:55 hover2_train_sentret.jsonl
-rw------- 1 root root  15M Oct 27 04:50 hover2_train_sentret_mh.jsonl
-rw------- 1 root root 3.2M Oct 27 03:13 hover2_val_docret.jsonl
-rw------- 1 root root 414K Oct 27 04:09 hover2_val_docret_mh.jsonl
-rw------- 1 root root 612K Oct 21 06:22 